# Interactive Heated Water Tank Digital Twin

This notebook is based on the supplied project report. Use the sliders to change the starting temperature, target temperature, room temperature, heater power, simulation duration, and simulated sensor noise. The model and graphs update when you change a value.

**Important:** This is a simulation-based prototype. It creates synthetic sensor readings and does not connect to or control a real water tank. Leave the default settings unchanged to reproduce the report's main scenario.


## How it works
1. You select the model inputs using the controls below.
2. The simulated sensor reading is the calculated tank temperature plus random noise.
3. The controller turns the simulated heater ON below the target and OFF at or above it.
4. The heat-balance equation updates the temperature every 10 simulated seconds.
5. The notebook displays summary values and plots for that scenario.


## Step 1 — Import libraries
NumPy performs the numerical calculations, Matplotlib draws the plots, and ipywidgets provides interactive input controls.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sliders are used where ipywidgets is available (for example, in Colab/Jupyter).
# A text-input fallback is included so the notebook can still run without widgets.
try:
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

THERMAL_CAPACITY = 420_000       # J/K
HEAT_LOSS_COEFFICIENT = 25       # W/K
TIME_STEP_SECONDS = 10           # seconds
RANDOM_SEED = 0


## Step 2 — Change the inputs
Move any slider to try a different scenario. The default values match the original report's main settings.


In [7]:
def run_and_display(initial_temperature=20.0, target_temperature=55.0,
                    ambient_temperature=20.0, heater_power=3000,
                    simulation_hours=3, sensor_noise_std=0.3):
    number_of_steps = int(simulation_hours * 3600 / TIME_STEP_SECONDS)
    rng = np.random.default_rng(RANDOM_SEED)

    temperature = float(initial_temperature)
    times_minutes = [0.0]
    tank_temperatures = [temperature]
    sensor_readings = [temperature + rng.normal(0, sensor_noise_std)]
    heater_states = [0]

    for step in range(number_of_steps):
        # Simulate an imperfect temperature sensor.
        measured_temperature = temperature + rng.normal(0, sensor_noise_std)

        # Feedback controller: heater ON below target, otherwise OFF.
        heater_on = 1 if measured_temperature < target_temperature else 0

        # Heat-balance model: heat added minus heat lost to the room.
        heat_added = heater_power * heater_on
        heat_lost = HEAT_LOSS_COEFFICIENT * (temperature - ambient_temperature)
        temperature = temperature + (TIME_STEP_SECONDS / THERMAL_CAPACITY) * (
            heat_added - heat_lost
        )

        times_minutes.append((step + 1) * TIME_STEP_SECONDS / 60)
        tank_temperatures.append(temperature)
        sensor_readings.append(measured_temperature)
        heater_states.append(heater_on)

    times = np.array(times_minutes)
    tank_temps = np.array(tank_temperatures)
    sensor_temps = np.array(sensor_readings)
    heater_states = np.array(heater_states)

    # Display numerical summary.
    recorded_temps = tank_temps[1:]
    control_steps = heater_states[1:]
    heater_on_percent = control_steps.mean() * 100 if len(control_steps) else 0
    heater_on_minutes = control_steps.sum() * TIME_STEP_SECONDS / 60

    print("HEATED WATER TANK DIGITAL TWIN — SIMULATION RESULTS")
    print("-" * 55)
    print(f"Starting temperature  : {tank_temps[0]:.2f} °C")
    print(f"Target temperature    : {target_temperature:.2f} °C")
    print(f"Ambient temperature   : {ambient_temperature:.2f} °C")
    print(f"Heater power          : {heater_power:.0f} W")
    print(f"Final tank temperature: {tank_temps[-1]:.2f} °C")
    print(f"Minimum temperature   : {recorded_temps.min():.2f} °C")
    print(f"Maximum temperature   : {recorded_temps.max():.2f} °C")
    print(f"Average temperature   : {recorded_temps.mean():.2f} °C")
    print(f"Heater ON             : {heater_on_percent:.1f}% of steps")
    print(f"Total heater ON time  : {heater_on_minutes:.1f} minutes")
    print(f"Duration              : {times[-1]:.0f} minutes")
    print(f"Sensor noise std. dev.: {sensor_noise_std:.2f} °C")
    print("\nNote: These are simulated values, not real tank measurements.")

    # Graph 1: temperature and the target.
    plt.figure(figsize=(11, 5))
    plt.plot(times, tank_temps, label="Simulated tank temperature", linewidth=2)
    plt.plot(times, sensor_temps, label="Simulated sensor reading", alpha=0.55, linewidth=1)
    plt.axhline(target_temperature, linestyle="--", linewidth=1.5,
                label=f"Target ({target_temperature:g} °C)")
    plt.title("Heated Water Tank: Temperature Monitoring")
    plt.xlabel("Time (minutes)")
    plt.ylabel("Temperature (°C)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Graph 2: simulated heater command.
    plt.figure(figsize=(11, 3))
    plt.step(times, heater_states, where="post")
    plt.yticks([0, 1], ["OFF", "ON"])
    plt.ylim(-0.2, 1.2)
    plt.title("Heater ON/OFF Control")
    plt.xlabel("Time (minutes)")
    plt.ylabel("Heater state")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def run_with_text_inputs():
    """Fallback: ask for values if interactive sliders are unavailable."""
    def ask(prompt, default):
        answer = input(f"{prompt} [{default}]: ").strip()
        return float(answer) if answer else float(default)

    print("Enter new values, or press Enter to keep the defaults.")
    initial = ask("Starting temperature in °C", 20.0)
    target = ask("Target temperature in °C", 55.0)
    ambient = ask("Room temperature in °C", 20.0)
    power = ask("Heater power in watts", 3000.0)
    hours = ask("Simulation duration in hours", 3.0)
    noise = ask("Sensor noise standard deviation in °C", 0.3)
    run_and_display(initial, target, ambient, power, int(hours), noise)


if WIDGETS_AVAILABLE:
    interact(
        run_and_display,
        initial_temperature=FloatSlider(value=20.0, min=0.0, max=50.0, step=1.0, description="Start °C", continuous_update=False),
        target_temperature=FloatSlider(value=55.0, min=25.0, max=85.0, step=1.0, description="Target °C", continuous_update=False),
        ambient_temperature=FloatSlider(value=20.0, min=0.0, max=40.0, step=1.0, description="Room °C", continuous_update=False),
        heater_power=IntSlider(value=3000, min=500, max=5000, step=250, description="Heater W", continuous_update=False),
        simulation_hours=IntSlider(value=3, min=1, max=6, step=1, description="Hours", continuous_update=False),
        sensor_noise_std=FloatSlider(value=0.3, min=0.0, max=1.0, step=0.1, description="Noise °C", continuous_update=False)
    );
else:
    print("Interactive sliders are unavailable here; using text input instead.")
    run_with_text_inputs()


interactive(children=(FloatSlider(value=20.0, continuous_update=False, description='Start °C', max=50.0, step=…

## How to demonstrate different temperatures
Try moving **Start °C** from 20 to 15, then to 40. Keep the target at 55°C and compare the final temperature and graphs. Try changing **Target °C** to 65°C, or increasing heater power to see how the warming behaviour changes. The widget reruns the simulation when a setting is changed.

The simulated time series is updated from the new inputs each time; this is not live data from a physical sensor.


## Conclusion
This interactive simulation demonstrates a heated water tank's temperature response, synthetic sensor noise, and basic ON/OFF feedback control. Interactive inputs make it possible to test multiple scenarios, while the underlying model remains the same as the one described in the report. It is a learning prototype only and is not connected to real equipment.
